In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

In [2]:
df = pd.read_csv(r"D:\Foundathon3.0\Mess-Food-Analytics\Final_Data\final_training_data.csv")
df = df[df['Plates'] > 0].reset_index(drop=True)
df.head()

,Mess,Month,Meal,Waste_KG,Plates,Waste_Percent,Estimated_Dish
0,D-MESS,Apr-23,BF,1006.0,24455,4,Idli/Dosa/Poha/Bread/Pongal/Chappathi/Vada/Upm...
1,D-MESS,Apr-23,LUNCH,3161.0,31425,10,Rice/Dal/Sabzi/Curd/Palak Panneer/Chicken Curr...
2,D-MESS,Apr-23,SNACKS,0.0,31350,0,"Tea/Biscuits/Samosa/Pav Baji,Pani Poori/Bonda"
3,D-MESS,Apr-23,DINNER,3175.0,31360,10,Roti/Paneer/Chicken/Rice/Jeera Dal/Panjabi Par...
4,D-MESS,May-23,BF,1023.0,24240,4,Idli/Dosa/Poha/Bread/Pongal/Chappathi/Vada/Upm...


In [3]:
# ─────────────────────────────────────────────
# 2. MEAL-WEIGHTED SKIP & WASTE RATES
#
# Based on real student behaviour:
#   BF     → highest skips (6:30–9AM, classes at 8AM, low motivation) + highest waste rate (food prepared assuming more turnout)
#   SNACKS → high skips (optional meal, students often off-campus)
#   DINNER → moderate-high skips (some eat outside)
#   LUNCH  → lowest skips (peak meal, everyone around)
# ─────────────────────────────────────────────

SKIP_RATES = {
    #  meal   : (min_skip%, max_skip%)   ← % of registered plates who skip
    'BF'      : (0.18, 0.30),   # 18–30% skip breakfast
    'SNACKS'  : (0.12, 0.22),   # 12–22% skip snacks
    'DINNER'  : (0.08, 0.16),   # 8–16%  skip dinner
    'LUNCH'   : (0.03, 0.09),   # 3–9%   skip lunch (lowest)
}

In [4]:
WASTE_RATES = {
    # meal    : (min_waste%, max_waste%)  ← additional waste ON TOP of skips
    # BF has highest waste — kitchen preps assuming more students than actually come
    'BF'      : (0.08, 0.15),   # 8–15% extra waste
    'DINNER'  : (0.05, 0.10),
    'LUNCH'   : (0.04, 0.08),
    'SNACKS'  : (0.02, 0.06),   # snacks waste is low (simpler items)
}

In [5]:
def simulate_skips(row):
    lo, hi   = SKIP_RATES[row['Meal']]
    # Academic phase modulates skips:
    # break months → fewer students on campus → even higher skip rates
    month = pd.to_datetime(row['Month'], format='%b-%y').month
    if month in [6, 1]:        # semester break
        lo, hi = min(lo * 1.4, 0.95), min(hi * 1.4, 0.95)
    elif month in [12, 5]:     # end of semester
        lo, hi = lo * 1.15, hi * 1.15
    rate = np.random.uniform(lo, hi)
    return int(round(row['Plates'] * rate))

def simulate_waste_plates(row):
    """Waste expressed as plate-equivalent lost (food prepped but not eaten)."""
    lo, hi = WASTE_RATES[row['Meal']]
    rate   = np.random.uniform(lo, hi)
    return int(round(row['Plates'] * rate))

In [6]:
df['total_skips']        = df.apply(simulate_skips,       axis=1)
df['waste_plate_equiv']  = df.apply(simulate_waste_plates, axis=1)

In [7]:
# Net plates = what was actually consumed
df['net_plates'] = (df['Plates'] - df['total_skips']).clip(lower=0)

print("="*55)
print("SKIP & WASTE SIMULATION SUMMARY")
print("="*55)
summary = df.groupby('Meal').agg(
    avg_registered   = ('Plates',           'mean'),
    avg_skips        = ('total_skips',      'mean'),
    avg_skip_pct     = ('total_skips',      lambda x: f"{(x / df.loc[x.index, 'Plates']).mean()*100:.1f}%"),
    avg_net_plates   = ('net_plates',       'mean'),
    avg_waste_equiv  = ('waste_plate_equiv','mean'),
).round(0)
print(summary.to_string())

SKIP & WASTE SIMULATION SUMMARY
        avg_registered  avg_skips avg_skip_pct  avg_net_plates  avg_waste_equiv
Meal                                                                           
BF             22847.0     5972.0        26.3%         16876.0           2585.0
DINNER         39766.0     4912.0        13.1%         34854.0           3175.0
LUNCH          38799.0     2375.0         6.4%         36423.0           2372.0
SNACKS         39638.0     6677.0        18.5%         32960.0           1500.0


In [8]:
dish_weights = {
    'biscuits'         : 0.80, 'bonda'            : 0.60,
    'bread'            : 0.90, 'chappathi'        : 0.50,
    'chicken'          : 0.70, 'chicken curry'    : 0.70,
    'curd'             : 0.70, 'dal'              : 0.80,
    'dal tadka'        : 0.80, 'dosa'             : 0.60,
    'idli'             : 0.80, 'jeera dal'        : 0.70,
    'kashmiri dum aloo': 0.70, 'palak panneer'    : 0.80,
    'paneer'           : 0.70, 'panjabi paratha'  : 0.50,
    'pav baji'         : 0.80, 'pani poori'       : 0.90,
    'poha'             : 0.90, 'pongal'           : 0.85,
    'poori'            : 0.70, 'potato masala'    : 0.67,
    'rice'             : 0.77, 'roti'             : 0.67,
    'sabzi'            : 0.70, 'samosa'           : 0.80,
    'steamed rice'     : 0.90, 'tea'              : 1.00,
    'upma'             : 0.60, 'vada'             : 0.80,
    'veg kuruma'       : 0.30, 'vegetable curry'  : 0.50,
}


In [9]:
def parse_dishes(dish_str):
    if not isinstance(dish_str, str):
        return []
    dishes = []
    for token in dish_str.split('/'):
        for sub in token.split(','):
            clean = sub.strip().lower()
            if clean:
                dishes.append(clean)
    return dishes

df['dish_list'] = df['Estimated_Dish'].apply(parse_dishes)

In [10]:
def menu_scores(dish_list):
    scores = [dish_weights.get(d, 0.5) for d in dish_list]
    if not scores:
        return 0, 0, 0, 0
    return round(np.mean(scores),4), round(np.max(scores),4), round(np.min(scores),4), round(np.sum(scores),4)

df[['menu_avg','menu_max','menu_min','menu_sum']] = df['dish_list'].apply(
    lambda d: pd.Series(menu_scores(d))
)

Time and academic data

In [11]:
df['month_parsed']  = pd.to_datetime(df['Month'], format='%b-%y')
df['month_num']     = df['month_parsed'].dt.month

def academic_phase(m):
    if m in [6, 1]:    return 0
    elif m in [7]:     return 1
    elif m in [5, 12]: return 2
    else:              return 3

df['academic_phase'] = df['month_num'].apply(academic_phase)
df['month_sin']      = np.sin(2 * np.pi * df['month_num'] / 12)
df['month_cos']      = np.cos(2 * np.pi * df['month_num'] / 12)

Encoding

In [12]:
mess_dummies      = pd.get_dummies(df['Mess'],  prefix='mess')
meal_dummies      = pd.get_dummies(df['Meal'],  prefix='meal')
df['mess_meal']   = df['Mess'] + '_' + df['Meal']
mess_meal_dummies = pd.get_dummies(df['mess_meal'], prefix='grp')

Splitting and training 

In [13]:
X = pd.concat([
    df[['academic_phase', 'month_sin', 'month_cos', 'month_num',
        'menu_avg', 'menu_max', 'menu_min', 'menu_sum',
        'total_skips']],          # ← NEW: skip count as input feature
    mess_dummies,
    meal_dummies,
    mess_meal_dummies,
], axis=1)

y = df['net_plates']

In [15]:
print(f"\nFeature matrix: {X.shape[1]} features, {X.shape[0]} rows")
print(f"   New features vs V2: +total_skips | Target: net_plates (not gross plates)")


Feature matrix: 43 features, 286 rows
   New features vs V2: +total_skips | Target: net_plates (not gross plates)


In [16]:

print("MODEL EVALUATION (5-Fold CV)")


models = {
    'Random Forest'    : RandomForestRegressor(n_estimators=300, min_samples_leaf=1, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42),
}

MODEL EVALUATION (5-Fold CV)


In [17]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    mae = -cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')
    r2  =  cross_val_score(model, X, y, cv=kf, scoring='r2')

    mape_scores = []
    for train_idx, val_idx in kf.split(X):
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = model.predict(X.iloc[val_idx])
        mape  = np.mean(np.abs((y.iloc[val_idx] - preds) / y.iloc[val_idx])) * 100
        mape_scores.append(mape)

    results[name] = {'MAE': mae.mean(), 'R2': r2.mean(), 'MAPE': np.mean(mape_scores)}
    print(f"\n{name}")
    print(f"  MAE  : {mae.mean():>10,.0f} ± {mae.std():,.0f} plates")
    print(f"  MAPE : {np.mean(mape_scores):>9.1f}%")
    print(f"  R²   : {r2.mean():>10.3f}")


Random Forest
  MAE  :      4,559 ± 851 plates
  MAPE :      30.1%
  R²   :      0.956

Gradient Boosting
  MAE  :      4,154 ± 535 plates
  MAPE :      27.4%
  R²   :      0.967


Hence we have reduced the MAPE score as well , while still increasing our R2 score 

In [19]:
best_name  = min(results, key=lambda k: results[k]['MAE'])
best_model = models[best_name]
best_model.fit(X, y)
print(f"\nBest model: {best_name}")


Best model: Gradient Boosting


Feature importance 

In [20]:

print("FEATURE IMPORTANCE (top 15)")

imp = pd.Series(best_model.feature_importances_, index=X.columns)
for feat, val in imp.sort_values(ascending=False).head(15).items():
    bar = '█' * int(val * 300)
    print(f"  {feat:<35} {val:.4f}  {bar}")

FEATURE IMPORTANCE (top 15)
  mess_SANNASI                        0.5576  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  academic_phase                      0.1291  ██████████████████████████████████████
  menu_sum                            0.0768  ███████████████████████
  total_skips                         0.0632  ██████████████████
  grp_SANNASI_BF                      0.0460  █████████████
  month_cos                           0.0293  ████████
  meal_BF                             0.0284  ████████
  month_sin                           0.0227  ██████
  month_num                           0.0184  █████
  menu_max                            0.0065  █
  meal_LUNCH                          0.0044  █
  mess_MEENAKSHI                      0.0039  █
  mess_STAFF MESS                     0.0033  █
  mess_J-MESS                         0.0026  
  menu_min              

In [21]:
print("\n" + "="*55)
print("ACTUAL vs PREDICTED")
print("="*55)
df['Predicted_Net'] = best_model.predict(X).round().astype(int)
df['Error_Pct']     = ((df['Predicted_Net'] - df['net_plates']) / df['net_plates'] * 100).round(1)

cols = ['Mess','Month','Meal','Plates','total_skips','net_plates','Predicted_Net','Error_Pct']
print(df[cols].head(24).to_string(index=False))
print(f"\nOverall avg error: {df['Error_Pct'].abs().mean():.1f}%")


ACTUAL vs PREDICTED
  Mess  Month   Meal  Plates  total_skips  net_plates  Predicted_Net  Error_Pct
D-MESS Apr-23     BF   24455         5501       18954          18839       -0.6
D-MESS Apr-23  LUNCH   31425         2735       28690          30618        6.7
D-MESS Apr-23 SNACKS   31350         6057       25293          27006        6.8
D-MESS Apr-23 DINNER   31360         4011       27349          26786       -2.1
D-MESS May-23     BF   24240         5540       18700          17861       -4.5
D-MESS May-23  LUNCH   29827         1350       28477          26952       -5.4
D-MESS May-23 SNACKS   32415         4690       27725          24709      -10.9
D-MESS May-23 DINNER   32415         5565       26850          27319        1.7
D-MESS Jun-23     BF    1360          480         880           1013       15.1
D-MESS Jun-23  LUNCH    1655          168        1487           1894       27.4
D-MESS Jun-23 SNACKS     840          144         696            783       12.5
D-MESS Jun-23 DINNE

In [22]:
def predict_net_plates(mess, meal, month_str, expected_skips):
    """
    Predict net plates after skips.

    mess            : e.g. 'D-MESS'
    meal            : e.g. 'LUNCH'
    month_str       : e.g. 'Aug-24'
    expected_skips  : int — how many students have opted out this month
    """
    dt = pd.to_datetime(month_str, format='%b-%y')
    m  = dt.month

    meal_dishes = df[df['Meal'] == meal]['dish_list'].iloc[0]
    scores      = [dish_weights.get(d, 0.5) for d in meal_dishes]

    row = pd.DataFrame([[0]*len(X.columns)], columns=X.columns)
    row['academic_phase'] = academic_phase(m)
    row['month_sin']      = np.sin(2 * np.pi * m / 12)
    row['month_cos']      = np.cos(2 * np.pi * m / 12)
    row['month_num']      = m
    row['menu_avg']       = np.mean(scores)
    row['menu_max']       = np.max(scores)
    row['menu_min']       = np.min(scores)
    row['menu_sum']       = np.sum(scores)
    row['total_skips']    = expected_skips

    for col in [f'mess_{mess}', f'meal_{meal}', f'grp_{mess}_{meal}']:
        if col in row.columns:
            row[col] = 1

    return max(0, int(round(best_model.predict(row)[0])))

In [23]:

print("SAMPLE PREDICTIONS")

tests = [
    ('D-MESS',  'BF',     'Aug-24', 4500),   # high skips — early morning
    ('D-MESS',  'LUNCH',  'Aug-24', 800),    # low skips  — peak meal
    ('D-MESS',  'SNACKS', 'Aug-24', 3200),   # moderate   — optional meal
    ('SANNASI', 'DINNER', 'Oct-24', 9000),
    ('SANNASI', 'BF',     'Jun-24', 8000),   # break month + BF = very low
]
print(f"\n  {'Mess':<12} {'Meal':<7} {'Month':<8} {'Skips':>8}  →  {'Net Plates':>10}")
print(f"  {'-'*12} {'-'*7} {'-'*8} {'-'*8}     {'-'*10}")
for mess, meal, month, skips in tests:
    pred = predict_net_plates(mess, meal, month, skips)
    print(f"  {mess:<12} {meal:<7} {month:<8} {skips:>8,}  →  {pred:>10,}")


SAMPLE PREDICTIONS

  Mess         Meal    Month       Skips  →  Net Plates
  ------------ ------- -------- --------     ----------
  D-MESS       BF      Aug-24      4,500  →      18,150
  D-MESS       LUNCH   Aug-24        800  →      27,306
  D-MESS       SNACKS  Aug-24      3,200  →      22,661
  SANNASI      DINNER  Oct-24      9,000  →     131,793
  SANNASI      BF      Jun-24      8,000  →      13,521
